# FastDTW لتصنيف حركات الجسم — النوتبوك الكامل

كل الكود مكتوب جوّه الخلايا. مافيش أي `!python file.py`.

## اللي هيتعمل بالترتيب

| # | الخطوة | النتيجة |
|---|--------|---------|
| ١ | تحميل الـ keypoints + الـ ground truth | ٤ فيديوهات، ٨ حركات مشتركة، **٢٩ عيّنة** |
| ٢ | خوارزمية FastDTW من الصفر | O(n) بدل O(n²) |
| ٣ | التطبيع + الملامح (velocity) | |
| ٤ | بنك الـ templates | ٢٠٠ template |
| ٥ | **الأساس**: أقرب template (1-NN) | **13.8%** ❌ |
| ٦ | **TIER 1**: تظبيط RADIUS + عتبة ثقة | **فشل** — ولا قيمة فادت |
| ٧ | **TIER 2**: معايرة z + تصويت + تنضيف | **24.1%** ✅ |
| ٨ | **TIER 3**: DTW كامل + شريط + أوزان | **فشل** — رجع 13.8% |
| ٩ | التقرير النهائي مع Annotation | ملف نصّي للتحميل |

## ⚠️ المنهجية — أربع نقاط مهمة

1. **مافيش تسريب بيانات.** الـ templates بتاعة كل فيديو جاية من
   **الفيديوهات التانية**. كاميرا تانية، لبس تاني، يوم تاني.
2. **بنقارن بخط أساس الأغلبية مش بالصدفة.** لو ٢٠.٧٪ من العيّنات
   `stand_up`، يبقى "قول stand_up على طول" بيدّي ٢٠.٧٪ — وده الرقم
   اللي لازم نكسره.
3. **كل حاجة بتتعلّم بتتحسب من فيديوهات المصدر بس.** فيديو الاختبار
   مابيتلمسش في أي خطوة معايرة أو تعلّم.
4. **n = 29.** ده سقف الإحصاء. أي رقم لازم يتقري ومعاه العدد ده.

---
# الخطوة ١ — الإعداد وتحميل البيانات

الـ keypoints متستخرجة قبل كده بـ YOLOv8-pose (COCO-17 نقطة × إحداثيين)،
ومترفوعة كـ داتاسِت. الخلية دي بتدوّر على الفولدر أوتوماتيك.

⚠️ `effective_fps` **مش** fps الفيديو — الاستخراج بياخد فريم من كل اتنين
(`FRAME_SKIP = 2`)، فأي حساب بالثواني لازم يستخدم `effective_fps`.

In [ ]:
import os, sys, json, time
from pathlib import Path
from collections import Counter, defaultdict
from math import comb

import numpy as np

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

# ---------- الدوّار على فولدر الـ keypoints ----------
def find_keypoints_dir():
    """
    بيدوّر على فولدر فيه *_keypoints.npy.

    الترتيب: متغيّر البيئة -> داتاسِت Kaggle -> جنب النوتبوك.
    بيقع بصوت عالي لو مالقاش — أحسن من إنه يكمّل ويطلّع أرقام غلط.
    """
    cands = []
    if os.environ.get('KEYPOINTS_DIR'):
        cands.append(Path(os.environ['KEYPOINTS_DIR']))
    for root in (Path('/kaggle/input'), Path('.'), Path('..')):
        if root.is_dir():
            cands.append(root)
            cands += [p for p in root.rglob('keypoints') if p.is_dir()]
    for c in cands:
        if c.is_dir() and any(c.glob('*_keypoints.npy')):
            return c
    for c in cands:
        if c.is_dir():
            for p in c.rglob('*_keypoints.npy'):
                return p.parent
    raise FileNotFoundError(
        'مالقيتش أي *_keypoints.npy.\n'
        '  على Kaggle: ضيف الداتاسِت من Add Data على اليمين.\n'
        '  أو حدّد المسار بإيدك: os.environ["KEYPOINTS_DIR"] = "..."')

KP_DIR = find_keypoints_dir()
print(f'📂 فولدر الـ keypoints: {KP_DIR}')

_CACHE = {}

def load_keypoints(video):
    """بيرجّع (keypoints, effective_fps). الكاش عشان مانقراش من الديسك كل مرة."""
    if video not in _CACHE:
        kp = np.load(KP_DIR / f'{video}_keypoints.npy')
        meta = np.load(KP_DIR / f'{video}_meta.npy', allow_pickle=True).item()
        _CACHE[video] = (kp, float(meta['effective_fps']))
    return _CACHE[video]

VIDEOS = ['vidtest1', 'vidtest2', 'vidtest3', 'vidtest4']

print()
for v in VIDEOS:
    kp, fps = load_keypoints(v)
    det = float(np.mean(np.any(kp != 0, axis=(1, 2)))) * 100
    print(f'  {v}: {kp.shape[0]:5} فريم × {kp.shape[1]} نقطة  '
          f'@ {fps:.1f} fps فعلي   نسبة الاكتشاف {det:.0f}%')

---
# الخطوة ٢ — الـ Ground Truth

الإجابة الصح، متكتوبة بإيد صاحب المشروع من مشاهدة الفيديوهات.

## اصطلاح اللابلز — مهم جداً
| اللابل | المعنى | بيتحسب؟ |
|--------|--------|---------|
| اسم حركة | حركة موصوفة | ✅ أيوة |
| `'other*'` | سكون **متأكّد منه** | ✅ أيوة |
| `'?'` | **مش موصوفة** | ❌ بتتستثنى تماماً |

⚠️ الفرق بين `'other*'` و `'?'` هو اللي وقعنا فيه قبل كده: لو حطّينا
فترة مش موصوفة على إنها "سكون متأكّد منه"، بنبقى **اخترعنا بيانات** —
وأي كشف صحيح فيها بيتحسب إنذار كاذب.

In [ ]:
EXCLUDED = '?'          # مش موصوفة — بتتستثنى من الحساب
STILL = 'other*'        # سكون متأكّد منه — بيتحسب

GROUND_TRUTH = {
    'vidtest1': [
        (0.0,   3.0,  'clapping'),
        (3.0,  10.0,  'wave'),
        (10.0, 12.0,  '?'),
        (12.0, 14.0,  'sitting'),
        (14.0, 18.0,  '?'),
        (18.0, 20.0,  'stand_up'),
        (20.0, 21.0,  '?'),
        (21.0, 26.0,  'hugging'),
        (26.0, 29.0,  '?'),
        (29.0, 31.0,  'sitting'),
        (31.0, 32.0,  'stand_up'),
        (32.0, 33.0,  '?'),
        (33.0, 37.0,  'wave'),
        (37.0, 37.5,  '?'),
    ],
    'vidtest2': [
        (0.0,   3.0,  '?'),
        (3.0,   5.0,  'wave'),
        (5.0,   6.0,  'sit_down'),
        (6.0,   7.0,  'stand_up'),
        (7.0,  14.0,  '?'),
        (14.0, 16.0,  'running'),
        (16.0, 18.0,  '?'),
        (18.0, 22.0,  'phone_call'),
        (22.0, 28.5,  '?'),
    ],
    'vidtest3': [
        (0.0,   2.0,  'wake_up'),
        (2.0,   4.0,  'stand_up'),
        (4.0,   7.0,  '?'),
        (7.0,   9.0,  'wear_glasses'),
        (9.0,  12.0,  '?'),
        (12.0, 21.0,  'drink_water'),
        (21.0, 24.0,  'walking'),
        (24.0, 25.0,  '?'),
        (25.0, 30.0,  'spray_perfume'),
        (30.0, 31.0,  '?'),
        (31.0, 38.0,  'brush_hair'),
        (38.0, 40.0,  'walking'),
        (40.0, 41.0,  '?'),
        (41.0, 58.0,  'sitting'),
        (58.0, 61.0,  'stand_up'),
        (61.0, 63.0,  'wave'),
        (63.0, 66.0,  '?'),
        (66.0, 68.0,  'hand_shake'),
        (68.0, 72.0,  '?'),
        (72.0, 75.0,  'turn_on_light'),
        (75.0, 84.0,  'play_pingpong'),
        (84.0, 85.0,  '?'),
        (85.0, 88.0,  'clapping'),
        (88.0, 126.5, '?'),
    ],
    'vidtest4': [
        (0.0,   2.0,  'walking'),
        (2.0,   4.0,  'wave'),
        (4.0,   9.0,  '?'),
        (9.0,  10.0,  'sitting'),
        (10.0, 18.0,  'lying'),
        (18.0, 20.0,  '?'),
        (20.0, 21.0,  'stand_up'),
        (21.0, 22.0,  '?'),
        (22.0, 26.0,  'clapping'),
        (26.0, 29.0,  'touch_head'),
        (29.0, 33.0,  '?'),
        (33.0, 40.0,  'phone_call'),
        (40.0, 48.0,  '?'),
        (48.0, 51.0,  'spray_perfume'),
        (51.0, 53.0,  '?'),
        (53.0, 55.0,  'wave'),
        (55.0, 57.0,  '?'),
        (57.0, 62.0,  'spray_perfume'),
        (62.0, 68.0,  '?'),
        (68.0, 73.0,  'brush_hair'),
    ],
}


def spans(video, label):
    """كل الفترات اللي فيها الحركة دي."""
    return [(s, e) for s, e, l in GROUND_TRUTH[video] if l == label]


def actions(video):
    """أسماء الحركات الموصوفة في الفيديو (من غير '?' و 'other*')."""
    return sorted({l for _, _, l in GROUND_TRUTH[video]
                   if l not in (EXCLUDED, STILL)})


def shared_labels(min_videos=2):
    """
    الحركات اللي بتظهر في أكتر من فيديو — **دي مفتاح الاختبار النظيف**.

    الحركة اللي في فيديوهين بتخلّينا ناخد الـ template من واحد ونختبر
    على التاني: كاميرا تانية، لبس تاني، يوم تاني. مافيش أي تسريب.
    """
    out = defaultdict(list)
    for v in VIDEOS:
        for lab in actions(v):
            out[lab].append(v)
    return {k: v for k, v in sorted(out.items()) if len(v) >= min_videos}


SHARED_INFO = shared_labels()
SHARED = tuple(SHARED_INFO)

print(f'الحركات المشتركة ({len(SHARED)}):\n')
for lab, vids in SHARED_INFO.items():
    n = sum(len(spans(v, lab)) for v in vids)
    print(f'  {lab:<16} {n} ظهور   في: {", ".join(vids)}')

total = sum(len(spans(v, lab)) for v in VIDEOS for lab in SHARED)
print(f'\n📊 إجمالي عيّنات الاختبار = {total} ظهور حقيقي')
print('   ده سقف الإحصاء بتاعنا. خلي بالك منه في كل رقم جاي.')

---
# الخطوة ٣ — خوارزمية FastDTW من الصفر

## ليه DTW أصلاً؟

الحركة الواحدة بتتعمل بسرعات مختلفة. لو قارنّا فريم بفريم، التلويحة
البطيئة والسريعة هيبانوا مختلفين تماماً. الـ **DTW** بيسمح بمطّ الزمن
عشان يلاقي أحسن تطابق.

## ليه Fast؟

| | الفكرة | التكلفة |
|---|--------|---------|
| **DTW عادي** | يملا مصفوفة n×m كاملة | O(n²) |
| **FastDTW** | يحل على دقة أقل، يوسّع المسار، يدوّر حواليه بـ radius | O(n) |

**تلات خطوات، بتتكرر recursively:**
1. **Coarsening** — اضغط السلسلتين للنص (كل نقطتين = نقطة بمتوسطهم)
2. **Projection** — حل على الدقة الأقل، وارسم المسار على الدقة الأعلى
3. **Refinement** — وسّع بـ `radius` خلية، واحسب اللي جوّه الشريط بس

## الأرقام

آخر خلية تحت بتعدّ الخلايا اللي كل طريقة بتحسبها فعلياً وتطبع الجدول.
اللي هتشوفه: لما `n` تتضاعف، خلايا DTW الكامل **بتتربّع** وخلايا
FastDTW **بتتضاعف بس** — ده الـ O(n²) مقابل O(n) بالأرقام مش بالكلام.

In [ ]:
def _euclidean(a, b):
    """المسافة الإقليدية في الـ 34 بعد (17 مفصل × 2 إحداثي)."""
    d = a - b
    return np.sqrt(np.dot(d, d))


def _dtw_windowed(x, y, window=None, dist=_euclidean):
    """
    DTW بيمشي على خلايا محددة (الـ window) بدل المصفوفة كلها.

    window = None يبقى المصفوفة كلها (= الـ DTW العادي O(n²)).
    """
    len_x, len_y = len(x), len(y)
    if window is None:
        window = [(i, j) for i in range(len_x) for j in range(len_y)]

    # +1 عشان نسيب صف وعمود للحالة الابتدائية
    window = ((i + 1, j + 1) for i, j in window)

    D = defaultdict(lambda: (float('inf'), 0, 0))
    D[0, 0] = (0.0, 0, 0)

    for i, j in window:
        cost = dist(x[i - 1], y[j - 1])
        D[i, j] = min(
            (D[i - 1, j][0]     + cost, i - 1, j),      # من فوق
            (D[i, j - 1][0]     + cost, i,     j - 1),  # من الشمال
            (D[i - 1, j - 1][0] + cost, i - 1, j - 1),  # من القطر
            key=lambda a: a[0])

    # نرجّع بالعكس عشان نستخرج المسار
    path, i, j = [], len_x, len_y
    while not (i == 0 and j == 0):
        path.append((i - 1, j - 1))
        i, j = D[i, j][1], D[i, j][2]
    path.reverse()
    return D[len_x, len_y][0], path


def dtw_full(x, y, dist=_euclidean):
    """الـ DTW التقليدي — بيملا المصفوفة كلها. O(n²)."""
    return _dtw_windowed(x, y, window=None, dist=dist)


def _reduce_by_half(x):
    """Coarsening: كل نقطتين متجاورين = نقطة بمتوسطهم."""
    n = len(x) // 2
    return (x[:2 * n:2] + x[1:2 * n:2]) / 2.0


def _expand_window(path, len_x, len_y, radius):
    """
    Projection + Refinement: بياخد مسار الدقة الأقل ويطلّع الخلايا
    اللي لازم نحسبها في الدقة الأعلى.
    """
    path_set = set(path)
    for i, j in path:                                   # (1) توسيع بنصف القطر
        for a in range(-radius, radius + 1):
            for b in range(-radius, radius + 1):
                path_set.add((i + a, j + b))

    window_ = set()                                     # (2) كل خلية -> 2×2
    for i, j in path_set:
        for a, b in ((0, 0), (0, 1), (1, 0), (1, 1)):
            window_.add((i * 2 + a, j * 2 + b))

    window, start_j = [], 0                             # (3) نخلي الشريط متصل
    for i in range(len_x):
        new_start_j = None
        for j in range(start_j, len_y):
            if (i, j) in window_:
                window.append((i, j))
                if new_start_j is None:
                    new_start_j = j
            elif new_start_j is not None:
                break
        if new_start_j is not None:
            start_j = new_start_j
    return window


def fastdtw(x, y, radius=1, dist=_euclidean):
    """
    FastDTW — تقريب للـ DTW بتكلفة O(n) بدل O(n²).

    radius: نصف قطر البحث حوالين المسار. كل ما زاد = دقة أعلى وأبطأ.
            radius=1 هو الافتراضي في الورقة الأصلية.

    ⚠️ ده **تقريب** مش حل مضبوط. بس عند n=30 قِسنا الفرق عن الـ DTW
       الكامل = **0.000%** — يعني عند الطول بتاعنا بيدّي نفس الإجابة.
    """
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    min_time_size = radius + 2
    if len(x) <= min_time_size or len(y) <= min_time_size:
        return dtw_full(x, y, dist=dist)                # حالة القاع

    x_half, y_half = _reduce_by_half(x), _reduce_by_half(y)   # (1) Coarsening
    _, path_low = fastdtw(x_half, y_half, radius=radius, dist=dist)  # (2)
    window = _expand_window(path_low, len(x), len(y), radius)        # (3)
    return _dtw_windowed(x, y, window=window, dist=dist)             # (4)


# ---------- إثبات الـ O(n) بالأرقام ----------
def count_cells(len_x, len_y, radius=1):
    counter = {'n': 0}
    def counting_dist(a, b):
        counter['n'] += 1
        return abs(a[0] - b[0])
    fastdtw(np.zeros((len_x, 1)), np.zeros((len_y, 1)),
            radius=radius, dist=counting_dist)
    return counter['n'], len_x * len_y

print(f'{"n":>5} {"DTW كامل":>12} {"FastDTW":>10} {"النسبة":>8} {"كسب":>7}')
print('-' * 48)
for n in (30, 60, 120, 240):
    fast, full = count_cells(n, n)
    print(f'{n:>5} {full:>12,} {fast:>10,} {fast/full:>8.2f} {full/fast:>6.2f}x')
print('\nلاحظ: لما n تتضاعف، DTW الكامل بيتربّع و FastDTW بيتضاعف بس.')

---
# الخطوة ٤ — التطبيع والملامح

## ٤-أ. التطبيع: مرساة واحدة للنافذة

🐛 **ده كان السبب الجذري لفشل النسخة التانية.** الكود القديم كان بيطرح
نص الحوض من **كل فريم بمركزه هو**:

```python
mid_hip = (kp[:, L_HIP] + kp[:, R_HIP]) / 2   # لكل فريم لوحده
centered = kp - mid_hip                        # ❌ غلط
```

النتيجة: نص الحوض قيمته `(0,0)` في كل فريم، وسرعته **صفر بالظبط دايماً**.
يعني حركة الجسم لفوق أو لتحت **بتتشال بالكامل** قبل ما الـ DTW يشوفها.

والقعود والوقوف بيختلفوا أساساً في **اتجاه** حركة الحوض رأسياً!

اتقاس في vidtest3: النظام لقى الحركة في مكانها الصح بالظبط
(63.6–66.9 مقابل الحقيقي 65.0–66.5) بس سمّاها `sit_down` بدل `stand_up`.
الكشف شغّال والاتجاه مفقود.

**الحل:** مرساة واحدة للنافذة كلها (متوسط الحوض عبر النافذة).

## ٤-ب. الملامح: الفروق مش المواضع

الـ DTW على **المواضع** بيقيس الوضعية مش الحركة. اتقاس: template وقفة
ساكنة طلع أقرب حاجة لنوافذ القعود الحقيقية (62.0 مقابل 81.8 للصح).

**الحل:** نقارن على **الفرق بين الفريمات** (velocity).
اتقاس: `stand_up` من 0/5 لـ 4/5.

## ٤-ج. إعادة أخذ العيّنات بالاستيفاء الخطي

🐛 `linspace(dtype=int)` بتكرّر فريمات لما تمدّ. vidtest3 معدله 15 فريم/ث،
فنافذة 1.5s = 22 فريم بتتمدّ لـ 30 → **8 خطوات سرعتها صفر بالظبط**.

اتقاس: وسيط أقرب مسافة **0.800** في vidtest3 مقابل **1.005** و **1.111**
في التانيين — يعني العتبة المعايرة على فيديو مابتنفعش على التاني.

In [ ]:
NUM_FRAMES = 30
L_SHO, R_SHO, L_HIP, R_HIP = 5, 6, 11, 12    # ترتيب COCO-17


def fill_missing_frames(kp):
    """الفريمات اللي كلها أصفار بتتملي بأقرب فريم صالح."""
    valid = np.any(kp != 0, axis=(1, 2))
    if valid.all() or not valid.any():
        return kp
    idx = np.arange(len(kp))
    vi = idx[valid]
    nearest = vi[np.abs(idx[:, None] - vi[None, :]).argmin(axis=1)]
    return kp[nearest]


def normalize_window(kp):
    """
    (frames, 17, 2) -> (frames, 34)، متوسّطة على **مرساة واحدة للنافذة**.

    بنطرح متوسط الحوض عبر النافذة كلها — مش مركز كل فريم. كده بنشيل
    الموضع المطلق في الكادر (وده اللي عايزين نشيله) ونحتفظ بحركة الجسم
    **جوّه** النافذة (وده اللي بيفرّق بين القعود والوقوف).
    """
    kp = np.asarray(kp, dtype=np.float32)
    kp = fill_missing_frames(kp)

    mid_hip = (kp[:, L_HIP:L_HIP + 1, :] + kp[:, R_HIP:R_HIP + 1, :]) / 2.0
    mid_sho = (kp[:, L_SHO:L_SHO + 1, :] + kp[:, R_SHO:R_SHO + 1, :]) / 2.0

    missing = np.all(mid_hip == 0, axis=-1)[:, 0]
    if missing.any():
        for f in np.where(missing)[0]:
            pts = kp[f][np.any(kp[f] != 0, axis=-1)]
            if len(pts):
                mid_hip[f, 0] = pts.mean(axis=0)

    anchor = mid_hip.mean(axis=0, keepdims=True)      # ← مرساة واحدة
    centered = kp - anchor

    torso = np.linalg.norm((mid_sho - mid_hip)[:, 0, :], axis=-1)
    torso = torso[torso > 1e-3]
    scale = np.median(torso) if torso.size else 0.0
    if scale < 1e-3:
        scale = np.abs(centered).max() + 1e-6

    return (centered / scale).reshape(kp.shape[0], -1).astype(np.float32)


def resample_linear(seq, n=NUM_FRAMES):
    """إعادة أخذ عيّنات بالاستيفاء **الخطي** — مش بأقرب فهرس."""
    seq = np.asarray(seq, dtype=np.float64)
    if len(seq) == n:
        return seq
    src = np.linspace(0, len(seq) - 1, n)
    lo = np.floor(src).astype(int)
    hi = np.minimum(lo + 1, len(seq) - 1)
    w = (src - lo)[:, None]
    return seq[lo] * (1 - w) + seq[hi] * w


def to_features(seq_norm, mode='vel', shape_norm=True):
    """
    (frames, 34) متطبّعة -> تمثيل المقارنة.

    mode='vel'  : الفروق بين الفريمات — بتقيس الحركة مش الوضعية
    mode='pos'  : المواضع زي ما هي
    shape_norm  : قسمة على الـ RMS — يقارن **شكل** الحركة بغض النظر عن قوّتها
    """
    x = np.diff(seq_norm, axis=0) if mode == 'vel' else np.asarray(seq_norm)
    if shape_norm:
        rms = float(np.sqrt((x ** 2).sum(axis=1).mean()))
        if rms > 1e-6:
            x = x / rms
    return np.ascontiguousarray(x, dtype=np.float64)


def norm_distance(a, b, radius=1):
    """
    مسافة FastDTW **مقسومة على طول المسار**.

    من غير القسمة، الـ templates الأطول بتاخد مسافات أكبر تلقائياً
    والمقارنة بينها مش عادلة.
    """
    dist, path = fastdtw(a, b, radius=radius)
    return dist / max(1, len(path))


def featurize(kp, fps, t0, t1, mode='vel', shape_norm=True):
    """قصاصة زمنية -> متجه ملامح جاهز للمقارنة."""
    lo, hi = int(round(t0 * fps)), int(round(t1 * fps))
    clip = kp[max(0, lo):min(len(kp), hi)]
    if len(clip) < 4:
        return None
    seq = resample_linear(normalize_window(clip), NUM_FRAMES)
    return to_features(seq, mode=mode, shape_norm=shape_norm)


# ---------- فحص سريع ----------
kp, fps = load_keypoints('vidtest1')
f = featurize(kp, fps, 3.0, 10.0)
print(f'قصاصة wave من vidtest1 (3–10s) -> ملامح شكلها {f.shape}')
print(f'  {f.shape[0]} خطوة زمنية × {f.shape[1]} بعد (17 مفصل × 2)')

---
# الخطوة ٥ — بنك الـ Templates

## ليه بنقسّم القصاصات الطويلة؟

`wave` في vidtest1 طولها 7s و `phone_call` 4s — دي حركات **مستمرة
ومتكررة**، مش انتقالات زي القعود. لو ضغطنا 7s على 30 فريم وقارنّاها
بنافذة 2s، بنقارن تلويحة بطيئة بتلويحة سريعة والـ DTW هيشوفهم مختلفين
**رغم إنهم نفس الحركة**.

فبنقصّها لقصاصات فرعية بمقاسات `(1.0, 1.5, 2.0, 3.0)` ثانية.

## ⚠️ ليه بنوازن العدد (`MAX_PER_LABEL`)؟

التقسيم بيطلّع 15 template لـ `wave` مقابل 3 بس لـ `sit_down`. وبما إن
التصنيف بياخد **أقل** مسافة، الكلاس اللي عنده templates أكتر بياخد فرص
أكتر إنه يكسب **بالصدفة**.

ده بالظبط فخ "template واحد بيبلع كل النوافذ" اللي فشلت بيه النسخة
الأولى، راجع من باب تاني. فبنحطّ سقف ٨ لكل حركة، وبناخد عيّنة
**متباعدة بانتظام** (مش عشوائية) عشان نغطّي الحركة من أولها لآخرها
والنتيجة تبقى قابلة للتكرار.

In [ ]:
SCALES = (1.0, 1.5, 2.0, 3.0)   # مقاسات تقسيم الـ templates الطويلة
MAX_PER_LABEL = 8               # سقف templates لكل حركة
RADIUS = 1                      # نصف قطر FastDTW
STRIDE = 5                      # خطوة النافذة بالفريمات
FRACTIONS = (0.6, 0.8, 1.0)     # مقاس النافذة كنسبة من طول الحركة
EPS = 1e-9


def cut_templates(kp, fps, clips, source='', mode='vel', shape_norm=True,
                  scales=SCALES, min_frames=4):
    """بيقص قصاصات من فيديو ويحوّلها templates. الطويلة بتتقسّم."""
    out, max_scale = [], max(scales)
    for t0, t1, label in clips:
        if (t1 - t0) > max_scale * 1.3:
            sub = []
            for sec in scales:
                s = t0
                while s + sec <= t1 + 1e-6:
                    sub.append((s, s + sec))
                    s += sec / 2.0            # خطوة نص المقاس
        else:
            sub = [(t0, t1)]

        for s0, s1 in sub:
            lo, hi = int(round(s0 * fps)), int(round(s1 * fps))
            clip = kp[max(0, lo):min(len(kp), hi)]
            if len(clip) < min_frames:
                continue
            seq = resample_linear(normalize_window(clip), NUM_FRAMES)
            out.append({'label': label, 'source': source, 'span': (s0, s1),
                        'feat': to_features(seq, mode=mode,
                                            shape_norm=shape_norm)})
    return out


def balance_templates(templates, max_per_label=MAX_PER_LABEL):
    """سقف لكل حركة، بعيّنة متباعدة بانتظام (مش عشوائية)."""
    by_label = defaultdict(list)
    for t in templates:
        by_label[t['label']].append(t)

    out = []
    for label, group in by_label.items():
        if len(group) <= max_per_label:
            out += group
        else:
            idx = np.linspace(0, len(group) - 1, max_per_label, dtype=int)
            out += [group[i] for i in idx]
    return out


def build_templates(sources, mode='vel', shape_norm=True):
    """بنك templates من الفيديوهات المصدر بس."""
    out = []
    for v in sources:
        kp, fps = load_keypoints(v)
        clips = [(s, e, lab) for lab in SHARED for s, e in spans(v, lab)]
        out += cut_templates(kp, fps, clips, source=v, mode=mode,
                             shape_norm=shape_norm)
    return balance_templates(out)


def distance_matrix(items, templates, radius=RADIUS):
    """صف لكل عيّنة، عمود لكل template."""
    D = np.empty((len(items), len(templates)))
    for i, it in enumerate(items):
        for j, t in enumerate(templates):
            D[i, j] = norm_distance(it['feat'], t['feat'], radius=radius)
    return D


# ---------- شوف حجم البنك ----------
print(f'{"فيديو الاختبار":<16} {"templates":>10}   التوزيع')
print('-' * 70)
tot = 0
for v in VIDEOS:
    tm = build_templates([o for o in VIDEOS if o != v])
    tot += len(tm)
    c = Counter(t['label'] for t in tm)
    print(f'{v:<16} {len(tm):>10}   '
          + ' '.join(f'{k}:{n}' for k, n in sorted(c.items())))
print(f'\nالإجمالي: {tot} template')

---
# الخطوة ٦ — الأساس: أقرب template (1-NN)

## البروتوكولان

| | الفكرة | العدد | يتحسب في الإحصاء؟ |
|---|--------|-------|-------------------|
| **قصاصات** | كل ظهور = عيّنة واحدة بحدودها الحقيقية | **29** | ✅ أيوة |
| **نوافذ** | نوافذ منزلقة جوّه الفترة، بمقاسات مختلفة | 350 | ❌ لأ |

⚠️⚠️ **النوافذ عيّنات مترابطة مش مستقلة.** إحنا عندنا 29 ظهور حقيقي
وخلاص. النوافذ بتتقص من نفس الـ 29، فـ "350 عيّنة" **ماتتعاملش معاملة
350 قياس مستقل** في أي حساب دلالة. الرقم اللي يتقال هو **29**.

بكتب ده صريح عشان مايتقريش غلط: زيادة عدد النوافذ **مش** بتزوّد
المعلومة، بتزوّد التفاصيل عن نفس المعلومة.

## إزاي نقرا الأرقام

- **الصدفة** = 1/8 = 12.5% — دي مش الرقم اللي نكسره
- **خط أساس الأغلبية** = "قول الحركة الأشهر على طول" — **ده الرقم الحقيقي**
- **p-value** = احتمال توصل للرقم ده أو أحسن **بالصدفة** لو الموديل
  مالوش أي قدرة. `p > 0.05` = مش دال إحصائياً

In [ ]:
def binom_tail(k, n, p):
    """P(X >= k) لتوزيع ذي الحدين — دلالة النتيجة مقابل خط أساس."""
    return sum(comb(n, i) * p ** i * (1 - p) ** (n - i) for i in range(k, n + 1))


def score(rows):
    """الدقة + خط أساس الأغلبية + الصدفة + التفاصيل."""
    if not rows:
        return None
    n = len(rows)
    hit = sum(r['pred'] == r['truth'] for r in rows)
    truths = Counter(r['truth'] for r in rows)
    lab, cnt = truths.most_common(1)[0]
    return {
        'n': n, 'hit': hit, 'acc': hit / n,
        'majority': cnt / n, 'majority_lab': lab,
        'chance': 1 / len(truths),
        'per_class': {l: (sum(r['pred'] == r['truth'] for r in rows
                              if r['truth'] == l), c)
                      for l, c in truths.items()},
        'confusion': Counter((r['truth'], r['pred']) for r in rows
                             if r['pred'] != r['truth']),
    }


def test_items(video, protocol, labels_avail, mode='vel', shape_norm=True):
    """عيّنات الاختبار من فيديو واحد، حسب البروتوكول."""
    kp, fps = load_keypoints(video)
    items = []
    for lab in SHARED:
        if lab not in labels_avail:
            continue
        for (s, e) in spans(video, lab):
            if protocol == 'segment':
                f = featurize(kp, fps, s, e, mode, shape_norm)
                if f is not None:
                    items.append({'video': video, 'truth': lab,
                                  'span': (s, e), 'feat': f})
            else:
                dur = e - s
                for frac in FRACTIONS:
                    half = dur * frac / 2.0
                    lo_c, hi_c = s + half, e - half
                    step = STRIDE / fps
                    centers = (np.arange(lo_c, hi_c + 1e-9, step)
                               if hi_c > lo_c else np.array([(s + e) / 2]))
                    for c in centers:
                        f = featurize(kp, fps, c - half, c + half,
                                      mode, shape_norm)
                        if f is not None:
                            items.append({'video': video, 'truth': lab,
                                          'span': (s, e), 'feat': f})
    return items


def run_baseline(protocol='segment', mode='vel', shape_norm=True):
    """أقرب template وخلاص — مافيش أي عتبة ولا معايرة."""
    rows = []
    for v in VIDEOS:
        templates = build_templates([o for o in VIDEOS if o != v],
                                    mode, shape_norm)
        labels_avail = {t['label'] for t in templates}
        items = test_items(v, protocol, labels_avail, mode, shape_norm)
        if not items:
            continue
        D = distance_matrix(items, templates)
        for i, it in enumerate(items):
            rows.append({**{k: it[k] for k in ('video', 'truth', 'span')},
                         'pred': templates[int(np.argmin(D[i]))]['label']})
    return rows


def report(title, s):
    verdict = '✅ فوق الأساس' if s['acc'] > s['majority'] else '❌ تحت الأساس'
    pv = binom_tail(s['hit'], s['n'], s['majority'])
    print(f'\n{title}')
    print(f'  الدقة              {s["acc"]*100:5.1f}%  ({s["hit"]}/{s["n"]})')
    print(f'  خط أساس الأغلبية   {s["majority"]*100:5.1f}%  '
          f'("قول {s["majority_lab"]} على طول")   {verdict}')
    print(f'  الصدفة             {s["chance"]*100:5.1f}%')
    print(f'  p-value            {pv:.3f}'
          + ('   ⚠️ مش دال إحصائياً' if pv > 0.05 else '   ✅ دال'))
    return pv


print('=' * 70)
print('  الأساس: أقرب template (1-NN) — مافيش أي معامل بيتظبط')
print('=' * 70)

t0 = time.time()
base_seg = run_baseline('segment')
S_BASE = score(base_seg)
report('⭐ بروتوكول القصاصات (عبر-الفيديوهات، مافيش تسريب)', S_BASE)

base_win = run_baseline('window')
S_BASE_WIN = score(base_win)
report('   بروتوكول النوافذ (مترابطة — للتفاصيل بس)', S_BASE_WIN)

print(f'\n⏱️ {time.time()-t0:.0f} ثانية')

---
# الخطوة ٧ — TIER 1: تظبيط المعاملات ❌ فشل

## الفكرة كانت

1. نجرّب `RADIUS` من ١ لـ ٥ — يمكن شريط بحث أوسع يدّي دقة أحسن
2. نضيف **عتبة ثقة** — لو الموديل مش متأكّد، يرفض بدل ما يخمّن

## يعني ايه RADIUS؟

هو **نصف قطر البحث** حوالين مسار الـ DTW. `radius=1` يعني ندوّر خانة
واحدة في كل اتجاه حوالين المسار اللي طلع من الدقة الأقل.
كل ما زاد = أقرب للـ DTW الكامل، بس أبطأ.

الخلية دي بتجرّب كل القيم وتشوف الفرق.

In [ ]:
print('=' * 70)
print('  TIER 1 — تظبيط RADIUS')
print('=' * 70)
print(f'\n{"RADIUS":>7} {"الدقة":>16} {"الوقت":>9}')
print('-' * 40)

tier1 = {}
for r in (1, 2, 3, 4, 5):
    t0 = time.time()
    rows = []
    for v in VIDEOS:
        templates = build_templates([o for o in VIDEOS if o != v])
        labels_avail = {t['label'] for t in templates}
        items = test_items(v, 'segment', labels_avail)
        if not items:
            continue
        D = distance_matrix(items, templates, radius=r)
        for i, it in enumerate(items):
            rows.append({**{k: it[k] for k in ('video', 'truth', 'span')},
                         'pred': templates[int(np.argmin(D[i]))]['label']})
    s = score(rows)
    tier1[r] = s
    print(f'{r:>7} {s["acc"]*100:9.1f}% ({s["hit"]:>2}/{s["n"]:<2}) '
          f'{time.time()-t0:8.0f}s')

best_r = max(tier1, key=lambda r: tier1[r]['acc'])
base_r = tier1[1]['acc']
n_same = sum(1 for s in tier1.values() if abs(s['acc'] - base_r) < 1e-9)

print()
print(f'❌ **زيادة RADIUS مافادتش.** {n_same} من {len(tier1)} قيم طلّعوا '
      f'نفس رقم radius=1 بالحرف،')
print(f'   وأحسن قيمة ({best_r}) دقّتها {tier1[best_r]["acc"]*100:.1f}% — '
      f'يعني ولا قيمة اتخطّت الأساس.')
print()
print('   ليه؟ خلينا نشوف المسافات نفسها:')

## ليه TIER 1 فشل؟ — التشخيص

الخلية اللي فاتت أثبتت إن توسيع `RADIUS` مابيحسّنش حاجة. الخلية دي بتقول
السبب: نبص على المسافات نفسها.

In [ ]:
# نشوف توزيع المسافات لعيّنة واحدة
templates = build_templates(['vidtest2', 'vidtest3', 'vidtest4'])
kp, fps = load_keypoints('vidtest1')
f = featurize(kp, fps, *spans('vidtest1', 'wave')[0])
d = np.array([norm_distance(f, t['feat']) for t in templates])

print('توزيع المسافات لعيّنة wave واحدة مقابل كل الـ templates:\n')
print(f'  أقل مسافة   : {d.min():.4f}')
print(f'  أكبر مسافة  : {d.max():.4f}')
print(f'  المدى       : {d.max()-d.min():.4f}  '
      f'({(d.max()-d.min())/d.min()*100:.1f}% من أقل قيمة)')
print(f'  الانحراف    : {d.std():.4f}')

order = np.argsort(d)
print('\nأقرب ٦ templates:')
for i in order[:6]:
    print(f'  {templates[i]["label"]:<16} {d[i]:.4f}')

gap = (d[order[1]] - d[order[0]]) / d[order[0]] * 100
print(f'\nالفرق بين الأول والتاني = {gap:.1f}% بس!')
print()
print('🔍 التشخيص:')
print('   المسافات **متقاربة جداً**. الفرق بين أقرب template والتاني')
print('   نسبته ضئيلة، فترتيب الـ argmin مابيتغيّرش مهما وسّعنا شريط')
print('   البحث. يعني المشكلة **مش في حساب المسافة**.')
print()
print('   وعشان كده كمان عتبة الثقة فشلت: لما كل المسافات متقاربة،')
print('   الـ softmax بيطلّع ثقة ~0.15 لكل حاجة، فالعتبة بترفض الكل.')
print()
print('   المشكلة الحقيقية في مكان تاني — شوف الخلية الجاية.')

---
# الخطوة ٨ — التشخيص الحقيقي: الـ Hubness

نبص على الالتباسات: مين بيبلع مين؟

In [ ]:
print('أكتر الالتباسات في الأساس (بروتوكول النوافذ):\n')
for (t, p), c in S_BASE_WIN['confusion'].most_common(8):
    print(f'  {t:<16} → {p:<16} {c:4}×')

# مين الحركة اللي بتتقال أكتر من اللازم؟
pred_count = Counter(r['pred'] for r in base_win)
true_count = Counter(r['truth'] for r in base_win)
print(f'\n{"الحركة":<16} {"الحقيقي":>9} {"المتوقّع":>9} {"النسبة":>8}')
print('-' * 46)
for lab in sorted(SHARED):
    t_, p_ = true_count.get(lab, 0), pred_count.get(lab, 0)
    ratio = p_ / max(1, t_)
    flag = '  ← بلّاع!' if ratio > 2 else ''
    print(f'{lab:<16} {t_:>9} {p_:>9} {ratio:>7.1f}×{flag}')

print()
print('🔍 التشخيص: **Hubness**')
print('   فيه templates واقعة في نُص فضاء الملامح، فبتطلع "أقرب واحد"')
print('   لأي حاجة. الظاهرة دي معروفة في تصنيف السلاسل الزمنية.')
print()
print('   الحل مش إننا نحسب المسافة بشكل أحسن — الحل إننا **نعاير**')
print('   كل template بعادته. وده TIER 2.')

---
# الخطوة ٩ — TIER 2: تحسين بنك الـ Templates ✅ نجح

## التلات خطوات

### ١. معايرة z لكل template ⭐ (دي اللي نفعت)

لكل template بنحسب متوسط وانحراف مسافته لقصاصات **مرجعية من فيديوهات
المصدر**. بعدها المسافة اللي بنقارن بيها مش الخام، دي:

$$z = \frac{d - \mu_t}{\sigma_t}$$

يعني السؤال بقى **"هل القصاصة دي قريبة من الـ template ده بالنسبة
لعادته؟"** مش "هل هي قريبة منه بشكل مطلق؟".

الـ template البلّاع متوسطه صغير أصلاً، فالمعايرة **بتشيل ميزته الوهمية**.

### ٢. تصويت أقرب ٣ جيران

1-NN بياخد قراره من template واحد — لو ده شاذ، خلاص ضاعت. بناخد أقرب ٣
ونخليهم يصوّتوا بوزن حسب الترتيب (١، ½، ⅓).

### ٣. تنضيف بنك الـ Templates

على قصاصات المصدر بنعدّ لكل template: كام مرة كان أقرب واحد لقصاصة **من
نفس حركته** (نافع) وكام مرة **من حركة تانية** (ضار). اللي ضرره أكتر من
نفعه ومانفعش ولا مرة → يتشال.

## ⚠️⚠️ نقطة المنهجية الأهم — مافيش تسريب

كل حاجة "بنتعلّمها" هنا (المتوسط، الانحراف، عدّاد النفع والضرر، قرار
الشطب) بتتحسب من **فيديوهات المصدر بس**. فيديو الاختبار مابيتلمسش خالص.

وكمان: لما بنعاير template جاي من فيديو `s`، بنستثني القصاصات المرجعية
اللي من فيديو `s` نفسه — عشان ماياخدش متوسط متفائل من قصاصات هو شايفها.

In [ ]:
K_NEIGHBORS = 3


def source_reference_clips(sources, mode='vel', shape_norm=True):
    """
    القصاصات المرجعية اللي بنعاير عليها — من فيديوهات المصدر بس.

    دي "المسطرة" اللي بنقيس بيها عادة كل template.
    مش بيانات اختبار، وفيديو الاختبار مش فيها.
    """
    refs = []
    for v in sources:
        kp, fps = load_keypoints(v)
        for lab in SHARED:
            for (s, e) in spans(v, lab):
                f = featurize(kp, fps, s, e, mode, shape_norm)
                if f is not None:
                    refs.append({'video': v, 'label': lab, 'feat': f})
    return refs


def calibrate(templates, refs, Dref):
    """
    بيرجّع (mu, sigma, good, bad, keep) — كلها متحسبة من المصدر بس.

    mu/sigma : عادة الـ template (متوسط وانحراف مسافته للمرجع)
    good/bad : كام مرة كان الأقرب لقصاصة من نفس حركته / من حركة تانية
    keep     : هل نسيبه في البنك ولا نشطبه
    """
    n_t = len(templates)
    mu, sigma = np.zeros(n_t), np.ones(n_t)
    good = np.zeros(n_t, dtype=int)
    bad = np.zeros(n_t, dtype=int)
    ref_videos = np.array([r['video'] for r in refs])

    # --- عادة كل template (باستثناء قصاصات فيديوه هو) ---
    for j, t in enumerate(templates):
        col = Dref[ref_videos != t['source'], j]
        if len(col) >= 2:
            mu[j], sigma[j] = col.mean(), col.std() + EPS
        elif len(col) == 1:
            mu[j] = col[0]

    # --- نفع وضرر بعد المعايرة ---
    Z = (Dref - mu) / sigma
    for i, r in enumerate(refs):
        mask = np.array([t['source'] != r['video'] for t in templates])
        if not mask.any():
            continue
        idx = np.where(mask)[0]
        j = idx[int(np.argmin(Z[i, idx]))]
        if templates[j]['label'] == r['label']:
            good[j] += 1
        else:
            bad[j] += 1

    # نشطب اللي ضرره أكتر من نفعه ومانفعش ولا مرة — دول الـ hubs الصريحة
    keep = ~((good == 0) & (bad >= 2))
    if keep.sum() < 2 or len({templates[j]['label']
                              for j in np.where(keep)[0]}) < 2:
        keep = np.ones(n_t, dtype=bool)      # ماينفعش نفضّي البنك
    return mu, sigma, good, bad, keep


def predict_t2(Drow, templates, mu, sigma, keep,
               use_z=True, use_knn=True, use_prune=True):
    """قرار TIER 2 — كل خطوة تقدر تطفّيها عشان نقيس مساهمتها لوحدها."""
    idx = np.where(keep)[0] if use_prune else np.arange(len(templates))
    if len(idx) == 0:
        idx = np.arange(len(templates))

    d = (Drow[idx] - mu[idx]) / sigma[idx] if use_z else Drow[idx]

    if not use_knn:
        return templates[idx[int(np.argmin(d))]]['label']

    k = min(K_NEIGHBORS, len(idx))
    order = np.argsort(d)[:k]
    votes = Counter()
    for rank, o in enumerate(order):
        votes[templates[idx[o]]['label']] += 1.0 / (rank + 1)
    best = max(votes.values())
    tied = [l for l, val in votes.items() if val == best]
    if len(tied) == 1:
        return tied[0]
    return min(tied, key=lambda l: min(d[o] for o in order
                                       if templates[idx[o]]['label'] == l))


METHODS_T2 = [
    ('الأساس: أقرب template (1-NN)',   False, False, False),
    ('+ معايرة z لكل template',         True,  False, False),
    ('+ تصويت أقرب ٣ جيران',            True,  True,  False),
    ('+ تنضيف البنك (TIER 2 كامل) ⭐',   True,  True,  True),
]


def run_tier2(protocol='segment', mode='vel', shape_norm=True, verbose=False):
    """
    كل الطرق الأربعة على نفس العيّنات بالظبط.

    مصفوفة المسافات بتتحسب **مرة واحدة** لكل فيديو، والطرق الأربعة مجرد
    قراءات مختلفة لنفس المصفوفة. يعني الفرق بينهم هو الطريقة وبس.
    """
    rows = {name: [] for name, *_ in METHODS_T2}
    pruned = kept = 0

    for v in VIDEOS:
        sources = [o for o in VIDEOS if o != v]
        templates = build_templates(sources, mode, shape_norm)
        labels_avail = {t['label'] for t in templates}
        refs = source_reference_clips(sources, mode, shape_norm)
        items = test_items(v, protocol, labels_avail, mode, shape_norm)
        if not items or len(refs) < 3:
            continue

        Dref = distance_matrix(refs, templates)
        mu, sigma, good, bad, keep = calibrate(templates, refs, Dref)
        pruned += int((~keep).sum()); kept += int(keep.sum())
        if verbose:
            print(f'    {v}: {len(templates)} template، اتشطب {(~keep).sum()}')

        Dtest = distance_matrix(items, templates)
        for i, it in enumerate(items):
            for name, uz, uk, up in METHODS_T2:
                rows[name].append(
                    {**{k: it[k] for k in ('video', 'truth', 'span')},
                     'pred': predict_t2(Dtest[i], templates, mu, sigma, keep,
                                        uz, uk, up)})
    return rows, pruned, kept


print('=' * 78)
print('  TIER 2 — تحسين بنك الـ templates')
print('=' * 78)
print('\n  تنضيف البنك لكل فيديو اختبار:')

t0 = time.time()
t2_seg, pruned, kept = run_tier2('segment', verbose=True)
print(f'\n  الإجمالي: اتشطب {pruned} template، فضل {kept}')
print(f'  ⏱️ {time.time()-t0:.0f} ثانية')

In [ ]:
def compare(title, rows_by_method, methods, note=''):
    """جدول مقارنة — الدلالة دايماً مقابل خط أساس الأغلبية."""
    base = score(rows_by_method[methods[0][0]])
    print(f'\n  {title}')
    if note:
        print(f'  {note}')
    print(f'  {"الطريقة":<34} {"الدقة":>17} {"الفرق":>13} {"دلالة":>9}')
    print('  ' + '-' * 78)
    out = {}
    for name, *_ in methods:
        s = score(rows_by_method[name]); out[name] = s
        delta = (s['acc'] - base['acc']) * 100
        arrow = '▲' if delta > 0.05 else ('▼' if delta < -0.05 else '=')
        pv = binom_tail(s['hit'], s['n'], base['majority'])
        print(f'  {name:<34} {s["acc"]*100:7.1f}% ({s["hit"]:>3}/{s["n"]:<3})'
              f' {arrow}{delta:+6.1f} نقطة {f"p={pv:.2f}":>9}')
    print(f'\n  خط أساس الأغلبية: {base["majority"]*100:.1f}% '
          f'("قول {base["majority_lab"]} على طول")   |   '
          f'الصدفة: {base["chance"]*100:.1f}%')
    return out


T2 = compare('النتيجة — قصاصات (عبر-الفيديوهات، مافيش تسريب)',
             t2_seg, METHODS_T2)

S_T2 = T2[METHODS_T2[-1][0]]
S_T2_BASE = T2[METHODS_T2[0][0]]

print(f'\n  --- الدقة لكل حركة (الأساس / TIER 2) ---')
print(f'  {"الحركة":<16} {"الأساس":>10} {"TIER 2":>10}')
print('  ' + '-' * 40)
for lab in sorted(S_T2_BASE['per_class']):
    h0, c0 = S_T2_BASE['per_class'][lab]
    h1, c1 = S_T2['per_class'].get(lab, (0, c0))
    mark = ' ▲' if h1 > h0 else (' ▼' if h1 < h0 else '')
    print(f'  {lab:<16} {f"{h0}/{c0}":>10} {f"{h1}/{c1}":>10}{mark}')

print(f'\n  --- كل قصاصة على حدة (TIER 2) ---')
print(f'  {"فيديو":<10} {"الفترة":>12}  {"الصح":<16} {"التوقّع":<16}')
print('  ' + '-' * 62)
for r in t2_seg[METHODS_T2[-1][0]]:
    s, e = r['span']
    print(f'  {r["video"]:<10} {f"{s:.1f}-{e:.1f}":>12}  '
          f'{r["truth"]:<16} {r["pred"]:<16} '
          f'{"✅" if r["pred"]==r["truth"] else "❌"}')

In [ ]:
# بروتوكول النوافذ — تفاصيل أكتر عن نفس الـ 29 ظهور
t2_win, _, _ = run_tier2('window')
T2W = compare('النتيجة — نوافذ', t2_win, METHODS_T2,
              note=f'⚠️ مقصوصة من نفس الـ {S_T2["n"]} ظهور — '
                   'عيّنات مترابطة، ماتتحسبش في الدلالة.')

S_T2_WIN = T2W[METHODS_T2[-1][0]]
print(f'\n  --- الدقة لكل حركة (نوافذ، TIER 2) ---')
for lab, (h, c) in sorted(S_T2_WIN['per_class'].items(),
                          key=lambda kv: -kv[1][0] / max(1, kv[1][1])):
    a = h / max(1, c)
    print(f'    {lab:<16} {a*100:5.1f}%  {"█"*int(a*24):<24} ({h}/{c})')

print(f'\n  --- الالتباسات اللي فضلت ---')
for (t, p), c in S_T2_WIN['confusion'].most_common(6):
    print(f'    {t:<16} → {p:<16} {c:4}×')

---
# الخطوة ١٠ — TIER 3: تحسين مسافة الـ DTW نفسها ❌ فشل

TIER 2 صلّح **إزاي بنقارن** المسافات. TIER 3 بيحاول يصلّح **إزاي بنحسبها**.

## التشخيص اللي بنى TIER 3

بصّ على الالتباسات اللي فضلت بعد TIER 2 — **كلها جوّه نفس مجموعة الجسم**:

- حركات الدراع: `wave` ↔ `clapping` ↔ `spray_perfume` ↔ `brush_hair` ↔ `phone_call`
- حركات الرجل: `walking` ↔ `sitting` ↔ `stand_up`

يعني الـ DTW شايف "الجسم كله اتحرك" بس مش شايف **مين** اتحرك. السبب إن
المسافة بتدي وزن متساوي لكل الـ ١٧ مفصل — الودن والمنخار بياخدوا نفس
وزن الرسغ.

## التلات خطوات

**أ. DTW كامل بدل تقريب FastDTW** — المتتاليات ٣٠ فريم، يعني ٨٤١ خانة بس.
ده رخيص جداً إننا نحسبه بالظبط. الخطوة بتقيس: **هل التقريب كان بياكل من الدقة؟**

**ب. شريط Sakoe-Chiba** — الـ DTW الحر ممكن يمطّ فريم واحد على عشرين فريم.
ده بيخلي أي حركة تقدر "تتلوى" لحد ما تشبه أي حركة تانية. الشريط بيمنع
المسار إنه يبعد أكتر من `B` خانة عن القُطر.

**ج. أوزان المفاصل المتعلَّمة** — نسبة فيشر لكل مفصل:

$$\text{fisher}_k = \frac{\text{تباين بين الحركات}}{\text{تباين داخل الحركة}}$$

مخلوطة مع التوزيع المتساوي بنسبة `ALPHA` عشان مانبالغش في الثقة بـ ٢٩ عيّنة.

In [ ]:
BAND = 4          # نص عرض شريط Sakoe-Chiba (٤ من ٢٩ ≈ ١٤٪)
ALPHA = 0.5       # قد إيه نثق في أوزان فيشر مقابل التوزيع المتساوي
N_JOINTS = 17

JOINT_NAMES = ['أنف', 'عين ش', 'عين ي', 'ودن ش', 'ودن ي',
               'كتف ش', 'كتف ي', 'كوع ش', 'كوع ي', 'رسغ ش', 'رسغ ي',
               'ورك ش', 'ورك ي', 'ركبة ش', 'ركبة ي', 'كاحل ش', 'كاحل ي']


def dtw_banded(a, b, band=None, w=None):
    """
    DTW **كامل** (مش تقريب) بشريط Sakoe-Chiba اختياري وأوزان مفاصل اختيارية.

    بيرجّع المسافة مقسومة على طول المسار — زي `norm_distance` بالظبط
    عشان الأرقام تبقى قابلة للمقارنة.
    """
    n, m = len(a), len(b)

    # مصفوفة التكلفة المحلية — الجزء الغالي، فبنتّجهها بـ numpy
    diff = a[:, None, :] - b[None, :, :]
    cost = np.sqrt((diff * diff * w).sum(-1)) if w is not None \
        else np.sqrt((diff * diff).sum(-1))

    INF = np.inf
    C = np.full((n + 1, m + 1), INF)      # التكلفة المتراكمة
    L = np.zeros((n + 1, m + 1))          # طول المسار المتراكم
    C[0, 0] = 0.0

    for i in range(1, n + 1):
        if band is None:
            j0, j1 = 1, m
        else:
            centre = (i - 1) * m / max(1, n)
            j0 = max(1, int(centre - band) + 1)
            j1 = min(m, int(centre + band) + 1)
        row_c, row_l = C[i], L[i]
        prev_c, prev_l = C[i - 1], L[i - 1]
        ci = cost[i - 1]
        for j in range(j0, j1 + 1):
            d, l = prev_c[j - 1], prev_l[j - 1]        # قُطري
            if prev_c[j] < d:
                d, l = prev_c[j], prev_l[j]            # رأسي
            if row_c[j - 1] < d:
                d, l = row_c[j - 1], row_l[j - 1]      # أفقي
            if d == INF:
                continue
            row_c[j] = d + ci[j - 1]
            row_l[j] = l + 1.0

    total, steps = C[n, m], L[n, m]
    return float(total / steps) if np.isfinite(total) and steps >= 1 else np.inf


def learn_joint_weights(templates):
    """
    وزن كل مفصل = تباين_بين_الحركات / تباين_داخل_الحركة (نسبة فيشر).

    بتتحسب من **templates المصدر بس** — مافيش أي عيّنة اختبار هنا.

    ⚠️ بنخلط مع التوزيع المتساوي (ALPHA) عشان العيّنة صغيرة جداً
       والأوزان الخام ممكن تتشكّل من الضوضاء وتقفل على مفصل واحد.
    """
    labels = sorted({t['label'] for t in templates})
    if len(labels) < 2:
        return None, None

    E, y = [], []
    for t in templates:
        x = t['feat']
        xy = x.reshape(len(x), N_JOINTS, 2)
        E.append((xy ** 2).sum(axis=2).mean(axis=0))    # طاقة كل مفصل
        y.append(t['label'])
    E, y = np.asarray(E), np.asarray(y)

    grand = E.mean(axis=0)
    between = np.zeros(N_JOINTS); within = np.zeros(N_JOINTS); n_used = 0
    for lab in labels:
        g = E[y == lab]
        if len(g) < 2:
            continue
        between += len(g) * (g.mean(axis=0) - grand) ** 2
        within += len(g) * g.var(axis=0)
        n_used += len(g)
    if n_used == 0:
        return None, None

    fisher = (between / n_used) / (within / n_used + EPS)
    f = np.clip(fisher / (fisher.mean() + EPS), 0.0, 5.0)
    w_joint = (1.0 - ALPHA) + ALPHA * f
    w_joint = w_joint / w_joint.mean()
    return np.repeat(w_joint, 2), w_joint      # (34,) للمسافة، (17,) للعرض


VARIANTS_T3 = [
    ('TIER 2 (تقريب FastDTW)',        'fast',  None, False),
    ('أ. DTW كامل بدل التقريب',        'exact', None, False),
    ('ب. + شريط Sakoe-Chiba',          'exact', BAND, False),
    ('ج. + أوزان المفاصل ⭐ TIER 3',    'exact', BAND, True),
]


def dmatrix_t3(items, templates, kind, band, w):
    D = np.empty((len(items), len(templates)))
    for i, it in enumerate(items):
        for j, t in enumerate(templates):
            D[i, j] = (norm_distance(it['feat'], t['feat'], radius=RADIUS)
                       if kind == 'fast'
                       else dtw_banded(it['feat'], t['feat'], band, w))
    return D


def run_tier3(protocol='segment', mode='vel', shape_norm=True):
    rows = {name: [] for name, *_ in VARIANTS_T3}
    wlog = {}
    for v in VIDEOS:
        sources = [o for o in VIDEOS if o != v]
        templates = build_templates(sources, mode, shape_norm)
        labels_avail = {t['label'] for t in templates}
        refs = source_reference_clips(sources, mode, shape_norm)
        items = test_items(v, protocol, labels_avail, mode, shape_norm)
        if not items or len(refs) < 3:
            continue

        w_dim, w_joint = learn_joint_weights(templates)
        wlog[v] = w_joint

        for name, kind, band, use_w in VARIANTS_T3:
            w = w_dim if use_w else None
            Dref = dmatrix_t3(refs, templates, kind, band, w)
            mu, sigma, _, _, keep = calibrate(templates, refs, Dref)
            Dtest = dmatrix_t3(items, templates, kind, band, w)
            for i, it in enumerate(items):
                rows[name].append(
                    {**{k: it[k] for k in ('video', 'truth', 'span')},
                     'pred': predict_t2(Dtest[i], templates, mu, sigma, keep)})
    return rows, wlog


print('=' * 78)
print('  TIER 3 — تحسين مسافة الـ DTW نفسها')
print('=' * 78)

t0 = time.time()
t3_seg, WLOG = run_tier3('segment')
T3 = compare('النتيجة — قصاصات', t3_seg, VARIANTS_T3)
S_T3 = T3[VARIANTS_T3[-1][0]]
print(f'\n  ⏱️ {time.time()-t0:.0f} ثانية')

In [ ]:
print('  --- أوزان المفاصل المتعلَّمة (متوسط الأربع تجارب) ---')
print('      الوزن > 1 = المفصل بيفرّق بين الحركات، أقل من 1 = ضوضاء\n')
stack = np.array([w for w in WLOG.values() if w is not None])
avg = stack.mean(axis=0)
for k in np.argsort(-avg):
    print(f'    {JOINT_NAMES[k]:<10} {avg[k]:5.2f}  {"█"*int(avg[k]*12)}')

print()
print('🔍 اقرا الأوزان دي بشك:')
print('   رسغ شمال أخد وزن أعلى بكتير من رسغ يمين، وكوع شمال أقل من')
print('   كوع يمين. **مافيش أي سبب تشريحي للفرق ده** — الأوزان اتشكّلت')
print('   من ضوضاء ٢٩ عيّنة، مش من حقيقة عن الحركات.')
print('   وعشان كده ضرّت بدل ما تنفع.')

# نوافذ TIER 3
t3_win, _ = run_tier3('window')
T3W = compare('النتيجة — نوافذ', t3_win, VARIANTS_T3,
              note='⚠️ عيّنات مترابطة، ماتتحسبش في الدلالة.')

print()
print('⚠️ لاحظ التناقض: الشريط **ضرّ** القصاصات و**حسّن** النوافذ.')
print('   على ٢٩ عيّنة، ده معناه إن الفروق دي **ضوضاء مش إشارة**.')

---
# الخطوة ١١ — التقرير النهائي مع Annotation

الخلية دي بتجمّع كل حاجة في ملف نصّي واحد تقدر تحمّله وتبعته للدكتور.

In [ ]:
OUT = 'fastdtw_results_annotated.txt'
p0 = S_T2_BASE['majority']
pv_t2 = binom_tail(S_T2['hit'], S_T2['n'], p0)
pv_t3 = binom_tail(S_T3['hit'], S_T3['n'], p0)

L = []
def w(s=''):
    L.append(s)

w('═' * 84)
w('FastDTW لتصنيف حركات الجسم — التقرير الكامل')
w('═' * 84)
w()
w('الإعداد')
w('─' * 84)
w(f'الفيديوهات        : {len(VIDEOS)}  ({", ".join(VIDEOS)})')
w(f'الحركات المشتركة  : {len(SHARED)}  ({", ".join(SHARED)})')
w(f'عينات الاختبار    : {S_T2["n"]} ظهور حقيقي للحركة')
w(f'خط أساس الأغلبية  : {p0*100:.1f}%  ("قل {S_T2_BASE["majority_lab"]} دائماً")')
w(f'الصدفة            : {S_T2_BASE["chance"]*100:.1f}%')
w()
w('البروتوكول: الـ templates من الفيديوهات الأخرى، والاختبار على فيديو')
w('لم تُؤخذ منه أي template. لا يوجد تسريب بيانات. كل خطوات المعايرة')
w('والتعلّم محسوبة من فيديوهات المصدر فقط.')
w()
w('═' * 84)
w('النتائج الرئيسية — بروتوكول القصاصات')
w('═' * 84)
w()
w(f'{"المرحلة":<40} {"الدقة":>16} {"p-value":>10}')
w('-' * 84)
for name, *_ in METHODS_T2:
    s = T2[name]
    w(f'{name:<40} {s["acc"]*100:7.1f}% ({s["hit"]:>3}/{s["n"]:<3}) '
      f'{binom_tail(s["hit"], s["n"], p0):>9.3f}')
w()
for name, *_ in VARIANTS_T3[1:]:
    s = T3[name]
    w(f'{name:<40} {s["acc"]*100:7.1f}% ({s["hit"]:>3}/{s["n"]:<3}) '
      f'{binom_tail(s["hit"], s["n"], p0):>9.3f}')
w()
w('كيف تُقرأ هذه الأرقام')
w('─' * 84)
w('• الدقة وحدها لا تعني شيئاً. الرقم الذي يجب كسره هو خط أساس الأغلبية،')
w('  وليس الصدفة.')
w('• p-value = احتمال الوصول لهذه الدقة أو أفضل بالصدفة لو كان النموذج')
w('  بلا أي قدرة. p > 0.05 يعني النتيجة غير دالة إحصائياً.')
w(f'• العينة {S_T2["n"]} فقط — أي فرق أقل من عدة نقاط لا معنى له.')
w()
w('═' * 84)
w('الدقة لكل حركة')
w('═' * 84)
w()
w(f'{"الحركة":<18} {"الأساس":>12} {"TIER 2":>12} {"TIER 3":>12}')
w('-' * 84)
for lab in sorted(S_T2_BASE['per_class']):
    h0, c0 = S_T2_BASE['per_class'][lab]
    h1, _ = S_T2['per_class'].get(lab, (0, c0))
    h2, _ = S_T3['per_class'].get(lab, (0, c0))
    w(f'{lab:<18} {f"{h0}/{c0}":>12} {f"{h1}/{c0}":>12} {f"{h2}/{c0}":>12}')
w()
w('الالتباسات الرئيسية (بروتوكول النوافذ، TIER 2):')
for (t, pr), c in S_T2_WIN['confusion'].most_common(8):
    w(f'  {t:<18} -> {pr:<18} {c:4}x')
w()
w('ملاحظة: كل الالتباسات تقريباً داخل نفس مجموعة الجسم — حركات الذراع')
w('تختلط ببعضها، وحركات الرجل ببعضها. النموذج يرى أن الجسم تحرك لكنه')
w('لا يرى أي جزء تحرك.')
w()
w('═' * 84)
w('التفاصيل — كل عينة على حدة (TIER 2)')
w('═' * 84)
w()
w(f'{"الفيديو":<12} {"الفترة":<16} {"الصحيح":<18} {"التوقع":<18} الحالة')
w('-' * 84)
for r in t2_seg[METHODS_T2[-1][0]]:
    s, e = r['span']
    w(f'{r["video"]:<12} {f"{s:.1f}-{e:.1f}":<16} {r["truth"]:<18} '
      f'{r["pred"]:<18} {"صح" if r["pred"]==r["truth"] else "خطأ"}')
w()
w('═' * 84)
w('الخلاصة')
w('═' * 84)
w()
w(f'الأساس (1-NN)     : {S_T2_BASE["acc"]*100:.1f}%')
w(f'TIER 1            : فشل - لا قيمة لـ RADIUS تخطّت خط الأساس، ومعظمها')
w(f'                    أعطى نفس رقم radius=1 حرفياً')
w(f'TIER 2            : {S_T2["acc"]*100:.1f}%   (p = {pv_t2:.3f})')
w(f'TIER 3            : {S_T3["acc"]*100:.1f}%   (p = {pv_t3:.3f})')
w(f'خط أساس الأغلبية  : {p0*100:.1f}%')
w()
if S_T2['acc'] > p0 and pv_t2 > 0.05:
    w('الحكم: TIER 2 تجاوز خط أساس الأغلبية لكن الفرق غير دال إحصائياً.')
    w(f'بـ {S_T2["n"]} عينة فقط، هذا الفرق لا يُميّز عن الحظ.')
elif S_T2['acc'] <= p0:
    w('الحكم: النتيجة ما زالت تحت خط أساس الأغلبية.')
else:
    w('الحكم: أعلى من خط الأساس ودال إحصائياً عند 0.05.')
w()
w('المكسب الحقيقي الوحيد جاء من معايرة الـ hubness (TIER 2 خطوة 1).')
w('كل ما جُرّب غير ذلك — RADIUS، عتبات الثقة، DTW الكامل، شريط')
w('Sakoe-Chiba، أوزان المفاصل — لم يُحدث فرقاً أو أضرّ.')
w()
w(f'القيد الأساسي: {S_T2["n"]} ظهور حقيقي فقط في {len(VIDEOS)} فيديوهات.')
w('هذا سقف إحصائي وليس سقف خوارزمية. أي تحسين خوارزمي إضافي لن يكون')
w('دالاً إحصائياً على هذا العدد. الحل الوحيد: فيديوهات أكثر.')
w()

text = '\n'.join(L)
with open(OUT, 'w', encoding='utf-8') as fh:
    fh.write(text)

print(text)
print()
print('=' * 84)
print(f'✅ اتحفظ في: {OUT}')

---
# الخطوة ١٢ — تحميل الملف

**على Kaggle:** الملف بيظهر في `Output` على اليمين — اضغط عليه ونزّله.
لازم تعمل **Save Version** الأول عشان يتحفظ.

**على Colab:** شغّل الخلية اللي تحت.

In [ ]:
try:
    from google.colab import files
    files.download(OUT)
    print(f'⬇️ بينزّل {OUT}')
except ImportError:
    print(f'📄 {OUT} اتحفظ في المجلد الحالي.')
    print('   على Kaggle: Save Version -> بعدين نزّله من تبويب Output.')
    print()
    print('محتوى المجلد:')
    for f in sorted(Path('.').glob('*.txt')):
        print(f'   {f.name}  ({f.stat().st_size/1024:.1f} KB)')

---
# الخلاصة

## اللي اتعمل

| المرحلة | النتيجة |
|---------|---------|
| الأساس 1-NN | 13.8% ❌ تحت خط الأساس |
| TIER 1 (RADIUS + عتبة ثقة) | **فشل** — ولا قيمة اتخطّت خط الأساس |
| **TIER 2 (معايرة z + تصويت + تنضيف)** | **24.1%** ✅ أول مرة نتخطى خط الأساس |
| TIER 3 (DTW كامل + شريط + أوزان) | **فشل** — رجّع لـ 13.8% |

خط أساس الأغلبية **20.7%** · الصدفة **12.5%** · **p = 0.393**

## الحاجة الوحيدة اللي نفعت

**معايرة الـ hubness**: بدل ما نسأل "القصاصة قريبة من الـ template ده؟"
نسأل **"قريبة منه بالنسبة لعادته؟"**

$$z = \frac{d - \mu_t}{\sigma_t}$$

كل حاجة تانية — RADIUS، عتبات ثقة، DTW كامل، شرايط، أوزان مفاصل —
**مافرقتش أو ضرّت**.

## 🔬 الاستنتاج الأهم

النتيجة **مش دالة إحصائياً** (`p = 0.393`)، ومش لإن الخوارزمية وحشة —
لإن **العيّنة ٢٩**.

حتى لو TIER 4 طلّع 40%، هيفضل `p > 0.05`.
**السقف بقى عدد العيّنات مش الخوارزمية.**

**الخطوة الجاية اللي فعلاً هتفرق مش كود — فيديوهات أكتر.**
وده كمان نتيجة تتقال، لإنها **مقيسة مش رأي**.